In [17]:
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import optuna

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
print(torch.cuda.get_device_name(0))

cuda
NVIDIA A100 80GB PCIe MIG 1g.10gb


In [19]:
class CHIMEFRBDataset(Dataset):
    def __init__(self, hdf5_path, catalog_path, target_length):
        self.hdf5_path = hdf5_path
        self.dt = 0.009830400085775182*1000

        self.target_length = target_length
        cat = pd.read_csv(catalog_path, low_memory=False)
        cat["repeater_name"] = cat["repeater_name"].fillna("").str.strip()
        self.repeater_set = set(
            cat.loc[cat["repeater_name"] != "", "tns_name"].str.strip()
        )
        with h5py.File(hdf5_path, "r") as f:
            self.keys = list(f.keys())

        
    @staticmethod
    def _pad_or_crop(wfall, target_length, center_idx):
        n_freq, n_time = wfall.shape
        half = target_length // 2
        start = center_idx - half
        end = start + target_length
        
        if start >= 0 and end <= n_time:
            return wfall[:, start:end]         
        
        src_start = max(start, 0)
        src_end = min(end, n_time)
        out = np.zeros((n_freq, target_length), dtype=wfall.dtype)
        dst_start = src_start - start
        out[:, dst_start:dst_start + (src_end - src_start)] = wfall[:, src_start:src_end]
        return out
    
    
    def __len__(self):
        return len(self.keys)
    
    
    def __getitem__(self, idx):
        key = self.keys[idx]
        
        with h5py.File(self.hdf5_path, "r") as f:
            wfall = f[key]["wfall_plot"][:]
            extent = np.array(f[key]["extent"])
            
        wfall = wfall.astype(np.float32)
        std = wfall.std(axis=1, keepdims=True)
        std[std == 0] = 1.0
        wfall = (wfall - wfall.mean(axis=1, keepdims=True)) / std

        peak = round(-extent[0] / self.dt)
        wfall = self._pad_or_crop(wfall, self.target_length, peak)
        tensor = torch.from_numpy(wfall)
        label = torch.tensor(int(key in self.repeater_set), dtype=torch.long)
        
        return tensor, label


def make_dataloader(
    hdf5_path: str,
    catalog_path: str,
    target_length: int,
    batch_size: int = 32,
    shuffle: bool = True,
    num_workers: int = 0,
    train_frac: float = 0.8,
    seed: int = 42,
):
    dataset = CHIMEFRBDataset(hdf5_path, catalog_path, target_length)

    n_total = len(dataset)
    n_train = int(n_total * train_frac)
    n_val = n_total - n_train

    generator = torch.Generator().manual_seed(seed)
    train_ds, val_ds = torch.utils.data.random_split(
        dataset, [n_train, n_val], generator=generator
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    labels = [dataset[i][1].item() for i in range(n_total)]
    n_rep = sum(labels)
    print(f"Dataset: {n_total} bursts | {n_rep} repeaters ({100*n_rep/n_total:.1f}%) "
          f"| {n_total-n_rep} non-repeaters")
    print(f"Train: {n_train} | Val: {n_val}")

    return train_loader, val_loader

TARGET_LENGTH = 128

train_loader, val_loader = make_dataloader(
    hdf5_path="/scratch/gpfs/MLISANTI/ra0438/all_bursts.hdf5",
    catalog_path="chimefrbcat2.csv",
    target_length=TARGET_LENGTH,
    batch_size=32,
    num_workers=0,
)

for wfall_batch, label_batch in train_loader:
    print(f"Batch shape : {wfall_batch.shape}")
    print(f"Labels      : {label_batch}")
    break

Dataset: 4536 bursts | 981 repeaters (21.6%) | 3555 non-repeaters
Train: 3628 | Val: 908
Batch shape : torch.Size([32, 256, 128])
Labels      : tensor([0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0,
        1, 0, 0, 1, 0, 0, 0, 0])


In [20]:
class SupConLoss(nn.Module):

    def __init__(self, temperature=0.07, contrast_mode='all',
                 base_temperature=0.07):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        
        if len(features.shape) < 3:
            raise ValueError('`features` needs to be [bsz, n_views, ...],'
                             'at least 3 dimensions are required')
        if len(features.shape) > 3:
            features = features.view(features.shape[0], features.shape[1], -1)

        batch_size = features.shape[0]
        if labels is not None and mask is not None:
            raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None:
            mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size:
                raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(device)
        else:
            mask = mask.float().to(device)

        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one':
            anchor_feature = features[:, 0] 
            anchor_count = 1
        elif self.contrast_mode == 'all':
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else:
            raise ValueError('Unknown mode: {}'.format(self.contrast_mode))

        anchor_dot_contrast = torch.div(
            torch.matmul(anchor_feature, contrast_feature.T),
            self.temperature)
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        mask = mask.repeat(anchor_count, contrast_count)
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size * anchor_count).view(-1, 1).to(device),
            0
        )
        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        mask_pos_pairs = mask.sum(1)
        mask_pos_pairs = torch.where(mask_pos_pairs < 1e-6, 1, mask_pos_pairs)
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs

        loss = - (self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss


class SinusoidalPE(nn.Module):
    def __init__(self, seq_len, embed_dim):
        super().__init__()
        pe = torch.zeros(seq_len, embed_dim)
        pos = torch.arange(seq_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, embed_dim, 2) * (-np.log(10000) / embed_dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe


class FRBMaskedAutoencoder(nn.Module):
    def __init__(self, seq_len, n_freq, embed_dim, contrast_dim=32, mask_ratio=0.25, dropout=0.1, n_heads=2, dim_feedforward=128):
        super().__init__()
        self.seq_len = seq_len
        self.n_freq = n_freq
        self.embed_dim = embed_dim
        self.contrast_dim = contrast_dim
        self.mask_ratio = mask_ratio

        self.enc_proj = nn.Linear(n_freq, embed_dim)
        self.enc_drop = nn.Dropout(dropout)
        self.enc_pe = SinusoidalPE(seq_len + 1, embed_dim)
        self.enc_blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(embed_dim, nhead=n_heads, dim_feedforward=dim_feedforward,
                                       batch_first=True, dropout=dropout) for _ in range(2)
        ])
        self.enc_norm = nn.LayerNorm(embed_dim)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.dec_pe = SinusoidalPE(seq_len + 1, embed_dim)
        self.dec_blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(embed_dim, nhead=n_heads, dim_feedforward=dim_feedforward,
                                       batch_first=True, dropout=dropout) for _ in range(2)
        ])
        self.dec_norm = nn.LayerNorm(embed_dim)
        self.dec_proj = nn.Linear(embed_dim, n_freq)

        self.cls_head = nn.Linear(embed_dim, 1)
        self.cls_drop = nn.Dropout(dropout)
        
        self.proj_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, contrast_dim),
        )

        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.mask_token, std=0.02)

    def mask_input(self, x):
        N, L, D = x.shape
        len_keep = int(L * (1 - self.mask_ratio))

        noise = torch.rand(N, L, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)

        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(x, dim=1,
                                index=ids_keep.unsqueeze(-1).expand(-1, -1, D))

        mask = torch.ones(N, L, device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)

        return x_masked, mask, ids_restore, ids_keep

    def encoder(self, x):
        x = self.enc_drop(self.enc_proj(x))
        x = x + self.enc_pe.pe[1:self.seq_len + 1] 
        x_vis, mask, ids_restore, ids_keep = self.mask_input(x)

        cls = (self.cls_token + self.enc_pe.pe[0]).expand(x_vis.size(0), -1, -1)
        x_vis = torch.cat([cls, x_vis], dim=1)

        for block in self.enc_blocks:
            x_vis = block(x_vis)
        x_vis = self.enc_norm(x_vis)

        return x_vis, mask, ids_restore

    def decoder(self, x_enc, ids_restore):
        B = x_enc.size(0)
        T = ids_restore.size(1)
        n_keep = x_enc.size(1) - 1                      
        mask_tokens = self.mask_token.expand(B, T - n_keep, -1)

        x_no_cls = x_enc[:, 1:, :]
        x_full = torch.cat([x_no_cls, mask_tokens], dim=1)
        x_full = torch.gather(x_full, dim=1,
                              index=ids_restore.unsqueeze(-1).expand(-1, -1, self.embed_dim))

        x_full = x_full + self.dec_pe.pe[1:T + 1]
        cls = x_enc[:, :1, :] + self.dec_pe.pe[0]
        x_full = torch.cat([cls, x_full], dim=1)

        for block in self.dec_blocks:
            x_full = block(x_full)
        x_full = self.dec_norm(x_full)

        recon = self.dec_proj(x_full[:, 1:, :])
        return recon

    def forward(self, x):
        x_t = x.permute(0, 2, 1)                       
        x_enc, mask, ids_restore = self.encoder(x_t)
        
        cls_token = x_enc[:, 0, :]
        cls_out = self.cls_drop(self.cls_head(cls_token))
        recon = self.decoder(x_enc, ids_restore) 
        
        proj = nn.functional.normalize(self.proj_head(cls_token), dim=1)
        proj = proj.unsqueeze(1)
        return recon, cls_out.squeeze(-1), mask, proj


SupConLoss_fn = SupConLoss(temperature=0.07, contrast_mode='all')

def compute_loss(cls_out, x_recon, mask, wfall, labels, proj,
                 alpha=1.0, beta=1.0, gamma=0.1, pos_weight=None, device='cpu'):
    pw = torch.tensor([pos_weight], dtype=torch.float32, device=device) if pos_weight else None
    cls_loss = nn.BCEWithLogitsLoss(pos_weight=pw)(cls_out, labels.float())
    target = wfall.permute(0, 2, 1)                   
    diff = (x_recon - target) ** 2
    recon_loss = (diff * mask.unsqueeze(-1)).sum() / (mask.sum() * wfall.size(1))
    con_loss = SupConLoss_fn(proj, labels=labels, mask=None)
    return alpha * cls_loss + beta * recon_loss + gamma * con_loss, cls_loss, recon_loss, con_loss

In [21]:
def train_one_epoch(model, loader, optimiser, device, alpha, beta, gamma, pos_weight_scalar):
    model.train()
    total, correct = 0, 0
    running_loss = running_cls = running_recon = running_con = 0.0

    for wfall, labels in loader:
        wfall, labels = wfall.to(device), labels.to(device)

        x_recon, cls_out, mask, proj = model(wfall)
        loss, cls_loss, recon_loss, con_loss = compute_loss(
            cls_out, x_recon, mask, wfall, labels, proj,
            alpha=alpha, beta=beta, gamma=gamma,
            pos_weight=pos_weight_scalar, device=device
        )

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        running_loss  += loss.item()
        running_cls   += cls_loss.item()
        running_recon += recon_loss.item()
        running_con   += con_loss.item()
        
        preds = (torch.sigmoid(cls_out.squeeze(-1)) > 0.5).long()
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    n = len(loader)
    print(f"  loss={running_loss/n:.4f}  cls={running_cls/n:.4f}  "
          f"recon={running_recon/n:.4f}  con={running_con/n:.4f}  acc={correct/total:.3f}")


torch.no_grad()
def evaluate(model, loader, device, alpha, beta, gamma, pos_weight_scalar):
    model.eval()
    total, correct = 0, 0
    running_loss = 0.0
    confusion_matrix_global = np.zeros((2, 2), dtype=int)

    for wfall, labels in loader:
        wfall, labels = wfall.to(device), labels.to(device)
        x_recon, cls_out, mask, proj = model(wfall)
        loss, *_ = compute_loss(
            cls_out, x_recon, mask, wfall, labels, proj,
            alpha=alpha, beta=beta, gamma=gamma,
            pos_weight=pos_weight_scalar, device=device
        )
        running_loss += loss.item()

        preds = (torch.sigmoid(cls_out.squeeze(-1)) > 0.5).long()
        cf_mat = confusion_matrix(labels.cpu(), preds.cpu(), labels=[0, 1])
        confusion_matrix_global += cf_mat
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        

    val_loss = running_loss / len(loader)
    val_acc = correct / total
    return val_loss, val_acc, confusion_matrix_global

In [24]:
N_EPOCHS = 150


best_overall = {"loss": float("inf")}

def objective(trial):
    embed_dim       = trial.suggest_categorical("embed_dim",       [32, 64, 128])
    contrast_dim    = trial.suggest_categorical("contrast_dim",    [16, 32, 64])
    mask_ratio      = trial.suggest_float("mask_ratio",            0.1, 0.75)
    dropout         = trial.suggest_float("dropout",               0.0, 0.5)
    n_heads         = trial.suggest_categorical("n_heads",         [1, 2, 4])
    dim_feedforward = trial.suggest_categorical("dim_feedforward", [64, 128, 256, 512])
    alpha           = trial.suggest_float("alpha",                 0.1, 5.0)
    beta            = trial.suggest_float("beta",                  0.1, 5.0)
    gamma           = trial.suggest_float("gamma",                 0.01, 1.0, log=True)
    pos_weight      = trial.suggest_float("pos_weight_scalar",     0.01, 0.5)
    lr_patience     = trial.suggest_int("LR_PATIENCE",             3, 15)
    es_patience     = trial.suggest_int("ES_PATIENCE",             5, 25)

    if embed_dim % n_heads != 0:
        raise optuna.exceptions.TrialPruned()

    model = FRBMaskedAutoencoder(
        seq_len=TARGET_LENGTH,
        n_freq=256,
        embed_dim=embed_dim,
        contrast_dim=contrast_dim,
        mask_ratio=mask_ratio,
        dropout=dropout,
        n_heads=n_heads,
        dim_feedforward=dim_feedforward,
    ).to(device)

    optimiser = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimiser, mode='min', factor=0.5, patience=lr_patience, min_lr=1e-6
    )

    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0
    
    print(f"Trial {trial.number}: {trial.params}")

    for epoch in range(N_EPOCHS):
        train_one_epoch(model, train_loader, optimiser, device,
                        alpha=alpha, beta=beta, gamma=gamma, pos_weight_scalar=pos_weight)
        val_loss, val_acc, confusion_matrix_global = evaluate(model, val_loader, device,
                            alpha=alpha, beta=beta, gamma=gamma, pos_weight_scalar=pos_weight)
        scheduler.step(val_loss)
        trial.report(val_loss, epoch)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= es_patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
        
    print(f"Trial {trial.number}: Final val_loss={best_val_loss:.4f}  val_acc={val_acc:.3f}")
    print("  Confusion Matrix:")
    print(confusion_matrix_global)

    if best_val_loss < best_overall["loss"]:
        best_overall["loss"] = best_val_loss
        torch.save({
            "model_state_dict": best_state,
            "params": trial.params,
            "val_loss": best_val_loss,
            "val_acc": val_acc
        }, "best_model.pt")
        print(f"New best model saved  (val_loss={best_val_loss:.4f}, trial={trial.number})")

    return best_val_loss


study = optuna.create_study(
    direction="minimize",
)
study.optimize(objective, n_trials=50)

print("\nBest trial:")
for k, v in study.best_trial.params.items():
    print(f"  {k}: {v}")

[I 2026-06-01 22:05:20,920] A new study created in memory with name: no-name-15f406a9-2f71-4245-a86e-25242914dd2d


Trial 0: {'embed_dim': 64, 'contrast_dim': 32, 'mask_ratio': 0.28795461306072234, 'dropout': 0.37321369290420514, 'n_heads': 4, 'dim_feedforward': 256, 'alpha': 0.3047390080883448, 'beta': 0.4206747819865325, 'gamma': 0.0338822179834633, 'pos_weight_scalar': 0.08348116376208907, 'LR_PATIENCE': 12, 'ES_PATIENCE': 19}
  loss=0.5970  cls=0.2663  recon=0.9438  con=3.5073  acc=0.782
  loss=0.5667  cls=0.2672  recon=0.8770  con=3.4336  acc=0.783
  loss=0.5606  cls=0.2653  recon=0.8641  con=3.4298  acc=0.783
  loss=0.5578  cls=0.2626  recon=0.8597  con=3.4269  acc=0.783
  loss=0.5578  cls=0.2675  recon=0.8561  con=3.4267  acc=0.783
  loss=0.5556  cls=0.2597  recon=0.8567  con=3.4249  acc=0.784
  loss=0.5552  cls=0.2607  recon=0.8550  con=3.4258  acc=0.783
  loss=0.5521  cls=0.2556  recon=0.8513  con=3.4246  acc=0.783
  loss=0.5529  cls=0.2554  recon=0.8538  con=3.4202  acc=0.785
  loss=0.5546  cls=0.2617  recon=0.8534  con=3.4179  acc=0.784
  loss=0.5538  cls=0.2644  recon=0.8501  con=3.4134 

[I 2026-06-01 22:11:22,637] Trial 0 finished with value: 0.4902914587793679 and parameters: {'embed_dim': 64, 'contrast_dim': 32, 'mask_ratio': 0.28795461306072234, 'dropout': 0.37321369290420514, 'n_heads': 4, 'dim_feedforward': 256, 'alpha': 0.3047390080883448, 'beta': 0.4206747819865325, 'gamma': 0.0338822179834633, 'pos_weight_scalar': 0.08348116376208907, 'LR_PATIENCE': 12, 'ES_PATIENCE': 19}. Best is trial 0 with value: 0.4902914587793679.


Early stopping at epoch 58
Trial 0: Final val_loss=0.4903  val_acc=0.835
  Confusion Matrix:
[[702  11]
 [139  56]]
New best model saved  (val_loss=0.4903, trial=0)
Trial 1: {'embed_dim': 128, 'contrast_dim': 64, 'mask_ratio': 0.19108950839559405, 'dropout': 0.19872275629766556, 'n_heads': 1, 'dim_feedforward': 512, 'alpha': 1.752842688224768, 'beta': 2.019705116749067, 'gamma': 0.10605569352649788, 'pos_weight_scalar': 0.1769418677288911, 'LR_PATIENCE': 15, 'ES_PATIENCE': 21}
  loss=2.6328  cls=0.2466  recon=0.9088  con=3.4423  acc=0.779
  loss=2.5117  cls=0.2263  recon=0.8676  con=3.4209  acc=0.784
  loss=2.4906  cls=0.2226  recon=0.8604  con=3.4210  acc=0.783
  loss=2.4733  cls=0.2205  recon=0.8539  con=3.4141  acc=0.783
  loss=2.4791  cls=0.2246  recon=0.8533  con=3.4134  acc=0.784
  loss=2.4545  cls=0.2139  recon=0.8508  con=3.4055  acc=0.785
  loss=2.4625  cls=0.2193  recon=0.8503  con=3.4013  acc=0.787
  loss=2.4647  cls=0.2220  recon=0.8490  con=3.4017  acc=0.784
  loss=2.4457 

[W 2026-06-01 22:14:27,217] Trial 1 failed with parameters: {'embed_dim': 128, 'contrast_dim': 64, 'mask_ratio': 0.19108950839559405, 'dropout': 0.19872275629766556, 'n_heads': 1, 'dim_feedforward': 512, 'alpha': 1.752842688224768, 'beta': 2.019705116749067, 'gamma': 0.10605569352649788, 'pos_weight_scalar': 0.1769418677288911, 'LR_PATIENCE': 15, 'ES_PATIENCE': 21} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/ra0438/.conda/envs/frb_env/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_3411392/3765489698.py", line 46, in objective
    train_one_epoch(model, train_loader, optimiser, device,
  File "/tmp/ipykernel_3411392/455370196.py", line 6, in train_one_epoch
    for wfall, labels in loader:
  File "/home/ra0438/.conda/envs/frb_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 701, in __next__
    data = self._next_data()

KeyboardInterrupt: 